In [1]:
!pip install essentia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.2 MB/s eta 0:00:00


In [2]:
import json
import essentia.standard as es
import os
import essentia

In [3]:
def extract_transition_features(path):
    def pool_get(pool, name, default=None):
        if pool is None:
            return default
        try:
            return pool[name]
        except (KeyError, TypeError):
            return default

    def to_list(x):
        if x is None:
            return None
        # cannot understand these essentia vectors
        try:
            return [float(v) for v in x]
        except TypeError:
            return None

    aggr_pool = None
    frames_pool = None
    try:
        me = es.MusicExtractor(
            lowlevelStats=['mean', 'stdev'],
            rhythmStats=['mean'],
            tonalStats=['mean']
        )
        aggr_pool, frames_pool = me(path)
    except Exception:
        aggr_pool = None
        frames_pool = None

    bpm = pool_get(aggr_pool, 'rhythm.bpm')
    beat_times = pool_get(aggr_pool, 'rhythm.beats_position')
    if beat_times is None:
        beat_times = pool_get(frames_pool, 'rhythm.beats_position')
    beat_times = to_list(beat_times)

    key = pool_get(aggr_pool, 'tonal.key_key')
    scale = pool_get(aggr_pool, 'tonal.key_scale')
    key_strength = pool_get(aggr_pool, 'tonal.key_strength')

    loudness = pool_get(aggr_pool, 'lowlevel.average_loudness')
    danceability = pool_get(aggr_pool, 'rhythm.danceability')

    # Spectral features (for timbre matching)
    spectral_centroid = pool_get(aggr_pool, 'lowlevel.spectral_centroid.mean')
    spectral_rolloff = pool_get(aggr_pool, 'lowlevel.spectral_rolloff.mean')
    dissonance = pool_get(aggr_pool, 'lowlevel.dissonance.mean')

    # Rhythm complexity
    onset_rate = pool_get(aggr_pool, 'rhythm.onset_rate')

    # Beat loudness (useful for detecting drops/buildups)
    beats_loudness = pool_get(aggr_pool, 'rhythm.beats_loudness')
    beats_loudness = to_list(beats_loudness)

    audio = es.MonoLoader(filename=path)()

    # should implement fall backs here, not sure why this wont work
    if key is None or scale is None or key_strength is None:
        try:
            k, s, ks = es.KeyExtractor()(audio)
            key, scale, key_strength = k, s, float(ks)
        except Exception:
            pass

    if bpm is None or beat_times is None or len(beat_times) == 0:
        try:
            r_out = es.RhythmExtractor2013(method='multifeature')(audio)
            bpm = float(r_out[0])
            beat_times = list(map(float, r_out[1])) if len(r_out) > 1 else []
        except Exception:
            bpm = None
            beat_times = []

    song_name = path.split('/')[-1]
    return {
        'song_name': song_name,
        'features': {
            "bpm": float(bpm) if bpm is not None else None,
            "key": key,
            "scale": scale,
            "key_strength": float(key_strength) if key_strength is not None else None,
            "loudness": float(loudness) if loudness is not None else None,
            "danceability": float(danceability) if danceability is not None else None,
            "spectral_centroid": float(spectral_centroid) if spectral_centroid is not None else None,
            "spectral_rolloff": float(spectral_rolloff) if spectral_rolloff is not None else None,
            "dissonance": float(dissonance) if dissonance is not None else None,
            "onset_rate": float(onset_rate) if onset_rate is not None else None,
        }
    }

In [4]:
path = "music"
results = []

for filename in os.listdir(path):
  if filename.endswith(".wav"):
    file_path = os.path.join(path, filename)
    output = extract_transition_features(file_path)
    results.append(output)

output_data = {"songs": results}

with open('results.json', 'w') as f:
  json.dump(output_data, f, indent=4)